In [3]:
import os
import json
import hashlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from anytree import Node, RenderTree

warnings.filterwarnings("ignore")

d:\git\Taxonomy_Buidling_Textual_Corpora\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df_train = pd.read_json("D:\git\Taxonomy_Buidling_Textual_Corpora\data\icecat_data_train.json")
df_train.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
1072689,ASUS,,https://images.icecat.biz/img/brand/thumb/161_...,ASUS,https://images.icecat.biz/img/brand/thumb/161_...,K31CD-IT049T,[],153,EN,PCs/Workstations,...,None,None,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
906402,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,686915-A41,[],2509,EN,Notebook Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...
411281,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,37745,[],953,EN,Fibre Optic Cables,...,None,None,NaN,None,None,None,None,NaN,2833>830>953,Computers & Electronics>Computer Cables>Fibre ...
425903,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,FA889AA#AC3,[],8194,EN,Handheld Mobile Computer Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8194,Computers & Electronics>Computers>Handheld Mob...
1047582,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,109559U,[],153,EN,PCs/Workstations,...,None,None,NaN,None,None,None,"[{'VirtualCategoryID': 194, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...


In [5]:
df_val = pd.read_json("D:\git\Taxonomy_Buidling_Textual_Corpora\data\icecat_data_validate.json")
df_val.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
309544,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,S26361-F5524-L800,[],1563,EN,Internal Solid State Drives,...,None,None,NaN,None,None,None,None,NaN,2833>206>2840>1563,Computers & Electronics>Data Storage>Data Stor...
805366,PanzerGlass,,https://images.icecat.biz/img/brand/thumb/1016...,PanzerGlass,https://images.icecat.biz/img/brand/thumb/1016...,PG1501,[],1568,EN,Screen Protectors,...,None,None,NaN,None,None,None,None,NaN,2833>107>1568,Computers & Electronics>Telecom & Navigation>S...
126809,2-Power,,https://images.icecat.biz/img/brand/thumb/1520...,2-Power,https://images.icecat.biz/img/brand/thumb/1520...,ALT268563B,[],911,EN,Memory Modules,...,None,None,NaN,None,None,None,None,NaN,2833>106>2844>911,Computers & Electronics>Computer Components>Sy...
922232,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,25205062,[],2509,EN,Notebook Spare Parts,...,None,None,NaN,None,None,None,None,NaN,2833>150>8355>2509,Computers & Electronics>Computers>Notebook Par...
567207,Verbatim,,https://images.icecat.biz/img/brand/thumb/669_...,Verbatim,https://images.icecat.biz/img/brand/thumb/669_...,97537,[],194,EN,Keyboards,...,None,None,NaN,None,None,None,None,NaN,2833>191>194,Computers & Electronics>Data Input Devices>Key...


In [6]:
df_test = pd.read_json("D:\git\Taxonomy_Buidling_Textual_Corpora\data\icecat_data_test.json")
df_test.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
500081,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,FSP:G-SW3Z560PRE0S,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...
741063,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,5KM83PA,None,151,EN,Notebooks,...,EN,13,1268560.0,EN,2018-10-21 21:55:51,"[Windows 10 Home 64-bit, Intel® Core™ i7-8565U...","[{'VirtualCategoryID': 329, 'UNCATID': '432115...",NaN,2833>150>151,Computers & Electronics>Computers>Notebooks
1091454,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,4NG33EA,[],153,EN,PCs/Workstations,...,EN,880,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
522928,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,83061,[],883,EN,Networking Cables,...,None,None,NaN,None,None,None,None,NaN,2833>830>883,Computers & Electronics>Computer Cables>Networ...
479478,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,5PS0A14091,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...


In [7]:
import pandas as pd
import re

TEXT_COLS = [
    "Brand",
    "ProductName",
    "Title",
    "Description.LongProductName",
    "Description.LongDesc",
    "SummaryDescription.LongSummaryDescription",
    "SummaryDescription.ShortSummaryDescription",
    "Category.Name.Value",
    "pathlist_names",
]

def to_text(x):
    if isinstance(x, list):
        x = " ".join(map(str, x))
    if x is None:
        return ""
    x = str(x)
    if x.lower() in ["none", "nan"]:
        return ""
    return x

def build_metadata_text(row):
    parts = [to_text(row.get(col, "")) for col in TEXT_COLS]
    return " ".join(p for p in parts if p)

def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"<[^>]+>", " ", s)              # remove HTML
    s = re.sub(r"[^a-z0-9\-+x/ ]+", " ", s)     # keep limited chars
    s = re.sub(r"\s+", " ", s).strip()
    return s


In [8]:


# take only 1000 rows to work with
df_train = df_train.sample(n=1000, random_state=42).reset_index(drop=True)

#  build metadata_text and metadata_text_clean
df_train["metadata_text"] = df_train.apply(build_metadata_text, axis=1)
df_train["metadata_text_clean"] = df_train["metadata_text"].apply(clean_text)

df_train[["metadata_text", "metadata_text_clean"]].head(2)


,metadata_text,metadata_text_clean
0,Acer 60.SH7N2.001 Acer 60.SH7N2.001 notebook s...,acer 60 sh7n2 001 acer 60 sh7n2 001 notebook s...
1,Tripp Lite Minicom Smart 108 Tripp Lite Minic...,tripp lite minicom smart 108 tripp lite minico...


In [9]:
df_val = df_val.sample(n=1000, random_state=42).reset_index(drop=True)

df_val["metadata_text"] = df_val.apply(build_metadata_text, axis=1)
df_val["metadata_text_clean"] = df_val["metadata_text"].apply(clean_text)

df_val[["metadata_text", "metadata_text_clean"]].head(2)


,metadata_text,metadata_text_clean
0,Lenovo 41Y8342 Lenovo 41Y8342 internal solid s...,lenovo 41y8342 lenovo 41y8342 internal solid s...
1,"HP 15-bs019ni HP 15-bs019ni Red,Black Notebook...",hp 15-bs019ni hp 15-bs019ni red black notebook...


In [10]:
df_test = df_test.sample(n=1000, random_state=42).reset_index(drop=True)

df_test["metadata_text"] = df_test.apply(build_metadata_text, axis=1)
df_test["metadata_text_clean"] = df_test["metadata_text"].apply(clean_text)

df_test[["metadata_text", "metadata_text_clean"]].head(2)


,metadata_text,metadata_text_clean
0,DELL 312-0106 DELL 312-0106 notebook spare par...,dell 312-0106 dell 312-0106 notebook spare par...
1,Xerox PHASER 6250DP ZW-KL LSR 24PPM 256MB 500V...,xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500v...


In [13]:
def make_ft_df(df):
    df_ft = df[["metadata_text_clean", "pathlist_names"]].copy()
    df_ft = df_ft.dropna(subset=["pathlist_names"])
    df_ft = df_ft[df_ft["metadata_text_clean"] != ""]
    df_ft = df_ft.rename(columns={
        "metadata_text_clean": "input_text",
        "pathlist_names": "target_path",
    })
    return df_ft.reset_index(drop=True)

df_train_ft = make_ft_df(df_train)
df_val_ft   = make_ft_df(df_val)
df_test_ft  = make_ft_df(df_test)

df_train_ft.head(3)



,input_text,target_path
0,acer 60 sh7n2 001 acer 60 sh7n2 001 notebook s...,Computers & Electronics>Computers>Notebook Par...
1,tripp lite minicom smart 108 tripp lite minico...,Computers & Electronics>Data Input Devices>KVM...
2,hp 826630-a41 hp 826630-a41 notebook spare par...,Computers & Electronics>Computers>Notebook Par...


In [14]:

df_val_ft.head(3)


,input_text,target_path
0,lenovo 41y8342 lenovo 41y8342 internal solid s...,Computers & Electronics>Data Storage>Data Stor...
1,hp 15-bs019ni hp 15-bs019ni red black notebook...,Computers & Electronics>Computers>Notebooks
2,gigabyte geforce 8400 256mb ddr2 gigabyte gefo...,Computers & Electronics>Computer Components>Sy...


In [15]:
df_test_ft.head(3)

,input_text,target_path
0,dell 312-0106 dell 312-0106 notebook spare par...,Computers & Electronics>Computers>Notebook Par...
1,xerox phaser 6250dp zw-kl lsr 24ppm 256mb 500v...,Computers & Electronics>Printers & Scanners>Pr...
2,startech com 5 ft cat 6 white molded rj45 utp ...,Computers & Electronics>Computer Cables>Networ...


In [16]:
import json
from pathlib import Path

out_dir = Path(r"D:\git\Taxonomy_Buidling_Textual_Corpora\fine_tuning\data")
out_dir.mkdir(parents=True, exist_ok=True)

def make_row(row):
    return {
        "messages": [
            {
                "role": "system",
                "content": "You are an assistant that assigns taxonomy paths to products.",
            },
            {
                "role": "user",
                "content": row["input_text"],
            },
            {
                "role": "assistant",
                "content": row["target_path"],
            },
        ]
    }

for name, df_ft in [("train", df_train_ft), ("val", df_val_ft), ("test", df_test_ft)]:
    out_path = out_dir / f"icecat_{name}_ft.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for _, r in df_ft.iterrows():
            f.write(json.dumps(make_row(r), ensure_ascii=False) + "\n")
    print(f"Saved {len(df_ft)} rows to {out_path}")


Saved 1000 rows to D:\git\Taxonomy_Buidling_Textual_Corpora\fine_tuning\data\icecat_train_ft.jsonl
Saved 1000 rows to D:\git\Taxonomy_Buidling_Textual_Corpora\fine_tuning\data\icecat_val_ft.jsonl
Saved 1000 rows to D:\git\Taxonomy_Buidling_Textual_Corpora\fine_tuning\data\icecat_test_ft.jsonl


In [17]:
file_path = r"D:\git\Taxonomy_Buidling_Textual_Corpora\fine_tuning\data\icecat_train_ft.jsonl"
with open(file_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline())


{"messages": [{"role": "system", "content": "You are an assistant that assigns taxonomy paths to products."}, {"role": "user", "content": "acer 60 sh7n2 001 acer 60 sh7n2 001 notebook spare part top case top case grey acer 60 sh7n2 001 type top case brand compatibility acer compatibility chromebook c710 acer 60 sh7n2 001 top case acer chromebook c710 notebook spare parts computers electronics computers notebook parts accessories notebook spare parts"}, {"role": "assistant", "content": "Computers & Electronics>Computers>Notebook Parts & Accessories>Notebook Spare Parts"}]}

{"messages": [{"role": "system", "content": "You are an assistant that assigns taxonomy paths to products."}, {"role": "user", "content": "tripp lite minicom smart 108 tripp lite minicom smart 108 kvm switch rack mounting black 100m vga ps/2 x 2 1600 x 1200 cat5 save space and money the smart 108 is a single-user analog cat5 kvm switch that gives you the ability to control multiple computers or servers from a single 

In [20]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import torch

train_path = r"D:\git\Taxonomy_Buidling_Textual_Corpora\fine_tuning\data\icecat_train_ft.jsonl"
val_path   = r"D:\git\Taxonomy_Buidling_Textual_Corpora\fine_tuning\data\icecat_val_ft.jsonl"

print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

train_ds = load_dataset("json", data_files=train_path)["train"]
val_ds   = load_dataset("json", data_files=val_path)["train"]

def messages_to_text(example):
    system_msg    = example["messages"][0]["content"]
    user_msg      = example["messages"][1]["content"]
    assistant_msg = example["messages"][2]["content"]

    full_text = (
        f"[SYSTEM] {system_msg}\n"
        f"[USER] {user_msg}\n"
        f"[ASSISTANT] {assistant_msg}"
    )
    return {"text": full_text}

train_ds = train_ds.map(messages_to_text)
val_ds   = val_ds.map(messages_to_text)

# 🔹 tiny model just to test pipeline
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    # ❗ important: labels = input_ids, so model can compute loss
    enc["labels"] = enc["input_ids"].copy()
    return enc

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

train_tok.set_format(type="torch")
val_tok.set_format(type="torch")

model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

training_args = TrainingArguments(
    output_dir="./taxonomy_tiny_test",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
)

trainer.train()


Using device: cpu


Map: 100%|██████████| 1000/1000 [00:00<00:00, 4207.39 examples/s]
                                       
  0%|          | 0/500 [04:12<?, ?it/s]        

{'loss': 10.8225, 'grad_norm': 0.5082921981811523, 'learning_rate': 4.9e-05, 'epoch': 0.02}


                                       
  0%|          | 0/500 [04:13<?, ?it/s]         

{'loss': 10.8214, 'grad_norm': 0.8793357014656067, 'learning_rate': 4.8e-05, 'epoch': 0.04}


...

                                       
  0%|          | 0/500 [05:00<?, ?it/s]          

{'loss': 10.7886, 'grad_norm': 0.4675247073173523, 'learning_rate': 0.0, 'epoch': 1.0}
























































































































































































                                       
                                              

  0%|          | 0/500 [05:23<?, ?it/s]          

                                       
100%|██████████| 500/500 [01:12<00:00,  6.89it/s]

{'eval_loss': 10.79064655303955, 'eval_runtime': 23.1415, 'eval_samples_per_second': 43.212, 'eval_steps_per_second': 21.606, 'epoch': 1.0}
{'train_runtime': 72.6061, 'train_samples_per_second': 13.773, 'train_steps_per_second': 6.886, 'train_loss': 10.80507633972168, 'epoch': 1.0}


TrainOutput(global_step=500, training_loss=10.80507633972168, metrics={'train_runtime': 72.6061, 'train_samples_per_second': 13.773, 'train_steps_per_second': 6.886, 'train_loss': 10.80507633972168, 'epoch': 1.0})

In [22]:
test_path = r"D:\git\Taxonomy_Buidling_Textual_Corpora\fine_tuning\data\icecat_test_ft.jsonl"

test_ds = load_dataset("json", data_files=test_path)["train"]
test_ds = test_ds.map(messages_to_text)   # reuse same function


Generating train split: 1000 examples [00:00, 102640.56 examples/s]
Map: 100%|██████████| 1000/1000 [00:00<00:00, 12808.01 examples/s]


In [24]:
from transformers import pipeline
import torch

# -----------------------------
# 1. Create text-generation pipeline
# -----------------------------
gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# -----------------------------
# 2. Load first example from test set
# -----------------------------
example = test_ds[0]   # test dataset must already be loaded earlier

system_msg = example["messages"][0]["content"]
user_msg   = example["messages"][1]["content"]
gold_label = example["messages"][2]["content"]

# -----------------------------
# 3. Build the prompt
# -----------------------------
prompt = (
    f"[SYSTEM] {system_msg}\n"
    f"[USER] {user_msg}\n"
    f"[ASSISTANT]"
)

print("=== PROMPT SENT TO MODEL ===")
print(prompt)

# -----------------------------
# 4. Generate model output
# -----------------------------
generated = gen(
    prompt,
    max_new_tokens=50,     # generate up to 50 new tokens
    do_sample=False        # deterministic output
)[0]["generated_text"]

print("\n=== MODEL RAW OUTPUT ===")
print(generated)

# -----------------------------
# 5. Extract only the predicted taxonomy path
# -----------------------------
predicted_taxonomy = generated.split("[ASSISTANT]", 1)[-1].strip()

print("\n=== MODEL PREDICTED TAXONOMY ===")
print(predicted_taxonomy)

print("\n=== GOLD (TRUE TAXONOMY) ===")
print(gold_label)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== PROMPT SENT TO MODEL ===
[SYSTEM] You are an assistant that assigns taxonomy paths to products.
[USER] dell 312-0106 dell 312-0106 notebook spare part battery 1900 mah 14 8 v always on the go- no more worries for running out of battery power you can back up your system with this 4-cell lithium-ion primary battery from dell it has an internal circuit board with chips that allow it to communicate with the notebook to monitor battery performance output voltage and temperature it also gives the notebook an accurate fuel gauge capability to determine how much battery runtime is left before the next recharge is required this product has been tested and validated on dell systems to ensure it will work with your computer and is compatible with dell latitude x300 notebook it is supported by dell technical support when used with a dell system highlights 28 whr capacity lets you work seamlessly while on the move provides very safe charging and discharging offers reliable power for dependable 